<a href="https://colab.research.google.com/github/lauandsalih/Multi-Armed-Bandit-for-Dynamic-Content-Allocation/blob/main/multi_armed_bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

class ThompsonSamplingBandit:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        # Initialize Beta distribution parameters (Prior: alpha=1, beta=1)
        self.alphas = np.ones(n_arms)
        self.betas = np.ones(n_arms)

    def select_arm(self):
        """
        Samples a probability from each arm's Beta distribution and
        returns the index of the arm with the highest sampled value.
        """
        samples = [np.random.beta(self.alphas[i], self.betas[i]) for i in range(self.n_arms)]
        return np.argmax(samples)

    def update(self, chosen_arm, reward):
        """
        Updates the Beta distribution parameters based on the observed reward (0 or 1).
        """
        if reward == 1:
            self.alphas[chosen_arm] += 1
        else:
            self.betas[chosen_arm] += 1

def simulate_environment(true_conversion_rates, n_trials=10000):
    """
    Simulates user traffic arriving at a website and testing both
    a traditional A/B test and a Multi-Armed Bandit.
    """
    n_arms = len(true_conversion_rates)
    bandit = ThompsonSamplingBandit(n_arms)

    # Trackers
    bandit_rewards = 0
    ab_test_rewards = 0
    bandit_selections = np.zeros(n_arms)
    ab_selections = np.zeros(n_arms)

    print(f"Running simulation with {n_trials} users...")
    print(f"Hidden True Conversion Rates: {true_conversion_rates}\n")

    for step in range(n_trials):
        # 1. Multi-Armed Bandit (Thompson Sampling)
        chosen_arm_bandit = bandit.select_arm()
        bandit_selections[chosen_arm_bandit] += 1

        # Simulate user converting or not based on true hidden probability
        reward_bandit = np.random.binomial(1, true_conversion_rates[chosen_arm_bandit])
        bandit.update(chosen_arm_bandit, reward_bandit)
        bandit_rewards += reward_bandit

        # 2. Traditional A/B Test (Equal Split)
        chosen_arm_ab = step % n_arms  # Round-robin traffic distribution
        ab_selections[chosen_arm_ab] += 1

        reward_ab = np.random.binomial(1, true_conversion_rates[chosen_arm_ab])
        ab_test_rewards += reward_ab

    # Calculate optimal possible rewards (if we knew the best arm from the start)
    best_rate = max(true_conversion_rates)
    optimal_rewards = best_rate * n_trials

    # Regret = Optimal Rewards - Actual Rewards
    bandit_regret = optimal_rewards - bandit_rewards
    ab_regret = optimal_rewards - ab_test_rewards

    # Print Results
    print("-" * 50)
    print("RESULTS AFTER 10,000 USERS")
    print("-" * 50)
    print(f"Traditional A/B Test Total Conversions: {ab_test_rewards}")
    print(f"Thompson Sampling Total Conversions:    {bandit_rewards}")
    print(f"Extra Conversions gained by using MAB:  {bandit_rewards - ab_test_rewards}\n")

    print("Traffic Distribution (How many users saw each variation):")
    for i in range(n_arms):
        print(f"Variation {i}: Bandit sent {int(bandit_selections[i])} users | A/B sent {int(ab_selections[i])} users")

if __name__ == "__main__":
    np.random.seed(42)
    # Let's say we have 3 website layouts. Layout 2 is the best.
    hidden_rates = [0.05, 0.08, 0.12]

    simulate_environment(hidden_rates, n_trials=10000)

Running simulation with 10000 users...
Hidden True Conversion Rates: [0.05, 0.08, 0.12]

--------------------------------------------------
RESULTS AFTER 10,000 USERS
--------------------------------------------------
Traditional A/B Test Total Conversions: 784
Thompson Sampling Total Conversions:    1179
Extra Conversions gained by using MAB:  395

Traffic Distribution (How many users saw each variation):
Variation 0: Bandit sent 55 users | A/B sent 3334 users
Variation 1: Bandit sent 242 users | A/B sent 3333 users
Variation 2: Bandit sent 9703 users | A/B sent 3333 users
